<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/CST2312/blob/main/Module07_Web_APIs_INSTRUCTOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CST2312 Module 7  Web APIs — INSTRUCTOR VERSION**

*This notebook is available at -- [bit.ly/cst2312cl16](https://bit.ly/cst2312cl16)*

> *includes exercises to be completed as a class assignment*

<div style="background:#fff3cd;border-left:6px solid #e0a800;padding:14px 18px;border-radius:4px;margin:12px 0;">
<strong>&#x1F4CB; INSTRUCTOR VERSION</strong><br>
This notebook contains worked solutions for all three exercises. Do not distribute to students. The student version (bit.ly/cst2312cl16) contains only <code># TODO</code> stubs in the exercise cells and placeholder comments in the solution cells.
</div>

### Required Reading

**PY4E Chapter 13 — Using Web Services** (required)  
[https://www.py4e.com/html3/13-web](https://www.py4e.com/html3/13-web)
  
### Supplementary Reading

**FOPP Chapter 24 — Internet APIs** (supplemental; free Runestone account required)  
[https://runestone.academy/ns/books/published/fopp/InternetAPIs/toctree.html](https://runestone.academy/ns/books/published/fopp/InternetAPIs/toctree.html)

---

## Learning Objectives

By the end of this module you will be able to:

- Explain what a REST API is and how it differs from a local function call
- Use Python's `requests` library to send HTTP GET requests and inspect the response
- Parse a JSON response into a Python dictionary and extract specific fields
- Distinguish between APIs that require an API key and those that do not
- Store API keys securely using the Colab Secrets tab
- Pass query parameters to an API using a Python dictionary and `requests.get()`
- Chain two APIs together so that the output of one becomes the input of the next
- Create a simple interactive map with the `folium` library using coordinates from an API response

## Connections to Prior Material

This module draws directly on several topics covered earlier in the course:

| Prior topic | How it appears here |
|---|---|
| **Dictionaries** (Module 11) | JSON responses parse into Python dictionaries; every field access uses `data["key"]` |
| **Functions** (earlier modules) | API calls are wrapped in reusable functions that accept parameters and return data |
| **Exception handling** (Module 09) | Network requests can fail; `try/except` blocks catch HTTP errors and timeouts |
| **f-strings** | Used to build URLs with variable components |

---

## Setup

Run this cell once before executing any other code in the notebook.

In [ ]:
import requests
import json
from google.colab import userdata   # Colab Secrets — used for API keys

---

## What Is an API?

A **Web API** (Application Programming Interface) is a service exposed over the internet
that accepts structured requests and returns structured data — typically JSON.
Calling a web API is conceptually identical to calling a Python function:

- The **URL** identifies the function (endpoint)
- **Query parameters** in the URL are the function's arguments
- The **JSON response** is the return value

The following URL is a complete API call you can paste into any browser:

[https://restcountries.com/v3.1/name/canada](https://restcountries.com/v3.1/name/canada)

Try it with different country names. Each response is a JSON document —
a structure that maps directly onto the Python dictionaries and lists you already know.

### JSON and Python Dictionaries

**JSON** (JavaScript Object Notation) is the standard format for data exchange between web services.
A JSON object is structurally identical to a Python dictionary:
key-value pairs, nested objects, and arrays that correspond to Python lists.

```json
{
  "name": {"common": "Canada"},
  "capital": ["Ottawa"],
  "population": 38005238,
  "region": "Americas"
}
```

When you call `response.json()`, the `requests` library parses this text and returns a Python
dictionary. Every field is then accessible with the familiar `data["key"]` syntax.

### Service-Oriented Architecture (SOA)

Most non-trivial applications do not do everything themselves.
They consume services provided by other applications:

- An e-commerce site calls a payment processor API to charge a credit card
- A travel site calls hotel reservation APIs to compare availability
- A financial dashboard calls a market data API to retrieve stock prices

Each service publishes an API — a documented contract specifying how requests must be formed
and what responses will look like. This design pattern is called **Service-Oriented Architecture (SOA)**.

REST (Representational State Transfer) is the dominant architectural style for web APIs today.

| Property | Meaning |
|---|---|
| **Stateless** | Each request contains all information needed; the server keeps no session state |
| **Resource-based** | Data is addressed by URL (e.g., `/v3.1/name/canada`) |
| **HTTP methods** | `GET` retrieves data; `POST` submits; `PUT` updates; `DELETE` removes |
| **Standard formats** | Responses in JSON (most common) or XML |

---

## Web APIs with Python — No API Key Required

Some APIs are open and require no authentication. They are useful for learning
because setup is immediate. We will use two:

- **`uselessfacts.jsph.pl`** — returns a random fact as JSON
- **`restcountries.com`** — returns geographic and demographic data for any country

### The `requests` Library and the Response Object

Two attributes of a `Response` object are most important:

| Attribute / Method | Description |
|---|---|
| `response.status_code` | Integer HTTP status (200 = success; 404 = not found; 5xx = server error) |
| `response.text` | Raw response body as a string |
| `response.json()` | Parses the response body and returns a Python dict or list |



---



### Example 1: Random Fact (uselessfacts.jsph.pl)

In [ ]:
# A simple GET request — no API key required
url = "https://uselessfacts.jsph.pl/api/v2/facts/random"

response = requests.get(url)

print("Status code:", response.status_code)
print()
print("Raw response text:")
print(response.text)

The raw response is a JSON string.
Calling `.json()` converts it into a Python dictionary so we can access individual fields.

In [ ]:
data = response.json()
print(type(data))   # confirms it is a dict
print()
print("Fact:  ", data["text"])
print("Source:", data["source"])

We can iterate through the keys in the resultant dictionarry to explore the data available.

In [ ]:
print(data.keys())

In [ ]:
print("The response information - ")
for key in data:
    print(f"\t{key}:  {data[key]}")



---



### Example 2: Country Data (restcountries.com)

`restcountries.com` returns a list containing one dictionary per matching country.
The response includes population, capital, region, languages, currency, flag, coordinates, and more.

In [ ]:
url = "https://restcountries.com/v3.1/name/canada"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()       # list of matching country dicts
    country = data[0]            # take the first result
    print("Country:   ", country["name"]["common"])
    print("Capital:   ", country["capital"][0])
    print("Population:", country["population"])
    print("Region:    ", country["region"])
else:
    print(f"Request failed with status code {response.status_code}")

Notice the nested access: `country["name"]["common"]` drills into a dictionary
nested inside another dictionary — exactly as with any Python dict.

The function below wraps this pattern so it can be called for any country name.
It also illustrates how `try/except` handles network or data errors gracefully
(a pattern introduced in Module 09).

In [ ]:
def get_country_info(country_name):
    """Return a summary dict of geographic and demographic data for a named country."""
    url = f"https://restcountries.com/v3.1/name/{country_name}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            country = response.json()[0]
            return {
                "name":       country["name"]["common"],
                "capital":    country["capital"][0],
                "population": country["population"],
                "region":     country["region"],
                "languages":  list(country.get("languages", {}).values()),
                "flag":       country.get("flag", "")
            }
        elif response.status_code == 404:
            return {"error": f"Country not found: '{country_name}'"}
        else:
            return {"error": f"HTTP {response.status_code}"}
    except Exception as e:
        return {"error": str(e)}

In [ ]:
country_name = input("Enter a country name: ")
info = get_country_info(country_name)
for key, value in info.items():
    print(f"{key:>12}: {value}")

---

## API Keys and Secure Storage

Most APIs that provide valuable or high-volume data require an **API key** —
a credential that identifies who is making each request. Keys enable providers to:

- Enforce **rate limits** (e.g., 1,000 requests/day on a free tier)
- Track usage for **billing**
- Revoke access if a key is misused

### Why API Keys Must Not Appear in Code

If a key is pasted directly into a notebook that is shared on Brightspace, pushed to GitHub,
or opened on another machine, anyone who reads the file can consume — or exhaust — your quota.

### Storing Keys in Colab Secrets

Click the **key icon** in the left sidebar to open the Secrets tab.
Add a name-value pair for each API key you use. Access it in code with:

```python
from google.colab import userdata
api_key = userdata.get("myAPIkeyName")
```

The string passed to `userdata.get()` must match the name in the Secrets tab exactly.

| Storage method | Persists after session? | Portable outside Google? |
|---|---|---|
| Colab Secrets tab | Yes | No |
| Google Secret Manager | Yes | No (requires GCP auth) |
| Encrypted local drive ([VeraCrypt](https://www.veracrypt.fr)) | Yes | Yes |

For this course, **Colab Secrets** is the recommended approach.

---

## Example: ipstack — IP Geolocation API

[ipstack.com](https://ipstack.com) returns the geographic location of any IP address:
country, region, city, latitude, longitude, ISP, and time zone.

**Setup:**
1. Register for a free key at [https://ipstack.com](https://ipstack.com)
2. In Colab Secrets, add a secret named `ipstackAPIkey`
3. Monitor your usage on the [ipstack dashboard](https://ipstack.com/dashboard)

URL pattern: `http://api.ipstack.com/{ip}?access_key={key}`  
Passing `check` instead of an IP address resolves the IP of the current Colab session.

<font color=blue><i>note: Be sure that the `Notebook Access` button is switched on in your Colab Secrets area when attempting to use a secret in a notebook.<i></font>

In [ ]:
api_key = userdata.get("ipstackAPIkey")

# 'check' resolves the IP of the current session
url = f"http://api.ipstack.com/check?access_key={api_key}"

response = requests.get(url)
print("Status code:", response.status_code)

In [ ]:
data = response.json()
print(data)

In [ ]:
print("IP address:", data["ip"])
print("City:      ", data["city"])
print("Country:   ", data["country_name"])
print("Latitude:  ", data["latitude"])
print("Longitude: ", data["longitude"])

<b>Why did your IP address not show as a local place?</b>  
<font color=gray>Answer: because your Colab session is running on a virtual machine with a connection to the internet, and an IP address, someplace else.</font>  

The function below wraps the pattern so it can be called with any IP address.
Notice it raises an `Exception` on failure rather than returning silently —
the calling code decides how to handle the error.

In [ ]:
def get_ip_location(ip_address, api_key):
    """Return a geolocation dict for the given IP address using the ipstack API."""
    url = f"http://api.ipstack.com/{ip_address}?access_key={api_key}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"API request failed: HTTP {response.status_code}")

In [ ]:
api_key = userdata.get("ipstackAPIkey")

location = get_ip_location("134.74.0.0", api_key)
print(f"City:        {location['city']}, {location['country_name']}")
print(f"Coordinates: {location['latitude']}, {location['longitude']}")

---

## API Calls with Query Parameters

Many APIs accept multiple parameters that control what data is returned.
Instead of building the URL manually, pass a Python dictionary to the `params` argument of
`requests.get()`. The library URL-encodes the parameters automatically.

```python
params = {"key1": "value1", "key2": "value2"}
response = requests.get(base_url, params=params)
# → base_url?key1=value1&key2=value2

### Example: OMDb — Movie Database API

[omdbapi.com](http://www.omdbapi.com) returns movie metadata from the IMDb database.

**Setup:**
1. Register for a free key at [http://www.omdbapi.com/apikey.aspx](http://www.omdbapi.com/apikey.aspx)
2. Store it in Colab Secrets as `omdbapiAPIkey`

**Key query parameters:**

| Parameter | Description |
|---|---|
| `t` | Exact movie title |
| `s` | Keyword search (returns a list) |
| `y` | Filter by release year |
| `plot` | `short` (default) or `full` |
| `apikey` | Your API key (required) |

In [ ]:
api_key = userdata.get("omdbapiAPIkey")

omdb_url = "http://www.omdbapi.com/"
parameters = {
    "t":      "Inception",
    "plot":   "short",
    "apikey": api_key
}

response = requests.get(omdb_url, params=parameters)
data = response.json()

print("Title:   ", data.get("Title"))
print("Year:    ", data.get("Year"))
print("Director:", data.get("Director"))
print("Rating:  ", data.get("imdbRating"))
print("Plot:    ", data.get("Plot"))

In [ ]:
# Prompt the user for any movie title
movie_title = input("Enter a movie title: ").strip()

parameters = {
    "t":      movie_title,
    "plot":   "short",
    "apikey": api_key
}

response = requests.get(omdb_url, params=parameters)
data = response.json()

if data.get("Response") == "True":
    print(f"\nTitle:    {data.get('Title')}")
    print(f"Year:     {data.get('Year')}")
    print(f"Director: {data.get('Director')}")
    print(f"Plot:     {data.get('Plot')}")
else:
    print(f"Not found: '{movie_title}'")

---

## Example: OpenWeatherMap

[openweathermap.org](https://openweathermap.org) provides current conditions and forecasts
for any location worldwide.

**Setup:**
1. Register for a free key at [https://openweathermap.org](https://openweathermap.org)
2. Store it in Colab Secrets as `openweathermapAPIkey`
3. Manage keys at the [OWM dashboard](https://home.openweathermap.org/api_keys)

The `units` parameter controls temperature scale: `imperial` (°F) or `metric` (°C).

In [ ]:
api_key = userdata.get("openweathermapAPIkey")

owm_url = "http://api.openweathermap.org/data/2.5/weather"
parameters = {
    "q":     "New York, NY, USA",
    "units": "imperial",
    "appid": api_key
}

response = requests.get(owm_url, params=parameters)
data = response.json()

print("City:        ", data["name"])
print("Temperature: ", data["main"]["temp"], "°F")
print("Feels like:  ", data["main"]["feels_like"], "°F")
print("Humidity:    ", data["main"]["humidity"], "%")
print("Wind speed:  ", data["wind"]["speed"], "mph")
print("Description: ", data["weather"][0]["description"])

The `params` dict approach also works when passing latitude and longitude
instead of a city name — useful for chaining with the ipstack API in Exercise 3.

In [ ]:
# Same request using lat/lon instead of city name
parameters_latlon = {
    "lat":   40.7128,
    "lon":  -74.0060,
    "units": "imperial",
    "appid": api_key
}

response = requests.get(owm_url, params=parameters_latlon)
data = response.json()
print(f"{data['name']}: {data['main']['temp']} °F, {data['weather'][0]['description']}")

---

## Mapping API Data with Folium

`folium` generates interactive maps (backed by Leaflet.js and OpenStreetMap).
A map is created from latitude and longitude coordinates; markers and popups
can be added. The result renders inline in Colab.

In [ ]:
import folium

# Map centered on Baruch College
baruch_map = folium.Map(
    location=[40.74074, -73.98325],
    zoom_start=15,
    control_scale=True,
    tiles="CartoDB positron"   # ← replaces "OpenStreetMap"
)

folium.Marker(
    location=[40.74074, -73.98325],
    icon=folium.Icon(icon="book", prefix="fa"),
    popup="Baruch College"
).add_to(baruch_map)

baruch_map

### Chaining APIs: Map a Location from an IP Address

The example below demonstrates **API chaining**: coordinates returned by ipstack
become the map center passed to folium.

In [ ]:
api_key = userdata.get("ipstackAPIkey")

location = get_ip_location("134.74.0.0", api_key)
lat = location["latitude"]
lon = location["longitude"]
city = location.get("city", "Unknown")

ip_map = folium.Map(
    location=[lat, lon],
    zoom_start=12,
    control_scale=True,
    tiles="CartoDB positron"   # ← replaces "OpenStreetMap"
)

folium.Marker(
    location=[lat, lon],
    icon=folium.Icon(icon="cloud", prefix="fa"),
    popup=f"{city} — IP location"
).add_to(ip_map)

ip_map

---

## Example: polygon.io — Financial Market Data API

[polygon.io](https://polygon.io) provides real-time and historical market data for
stocks, options, forex, and cryptocurrencies. It is one of the primary data sources
used in the financial analytics portion of this course.

**Setup:**
1. Register for a free key at [https://polygon.io](https://polygon.io)
2. Store it in Colab Secrets as `polygonAPIkey`
3. Review the full documentation at [https://polygon.io/docs/](https://polygon.io/docs/)

The endpoint below returns the previous trading day's closing data for any ticker.
Free-tier accounts have a short delay on market data.

In [ ]:
api_key = userdata.get("polygonAPIkey")

ticker = "AAPL"
url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/prev"
parameters = {
    "adjusted": "true",
    "apiKey":   api_key
}

response = requests.get(url, params=parameters)
data = response.json()

print("Status:", data.get("status"))
print("Ticker:", data.get("ticker"))

if data.get("resultsCount", 0) > 0:
    result = data["results"][0]
    print(f"Close:   ${result['c']:.2f}")
    print(f"Open:    ${result['o']:.2f}")
    print(f"High:    ${result['h']:.2f}")
    print(f"Low:     ${result['l']:.2f}")
    print(f"Volume:  {result['v']:,.0f}")

In [ ]:
# Look up any ticker
ticker = input("Enter a stock ticker symbol (e.g. MSFT, TSLA, AMZN): ").strip().upper()

url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/prev"
parameters = {"adjusted": "true", "apiKey": api_key}

response = requests.get(url, params=parameters)
data = response.json()

if data.get("status") == "OK" and data.get("resultsCount", 0) > 0:
    result = data["results"][0]
    print(f"\n{ticker} — Previous Trading Day")
    print(f"  Close:  ${result['c']:.2f}")
    print(f"  Open:   ${result['o']:.2f}")
    print(f"  High:   ${result['h']:.2f}")
    print(f"  Low:    ${result['l']:.2f}")
    print(f"  Volume: {result['v']:,.0f}")
else:
    print(f"No data found for ticker '{ticker}'")

---

## Exercises

### Exercise 1 — No API Key Required

Using the `restcountries.com` API (no key required), write code that:

a. Prompts the user for a country name  
b. Retrieves the country data  
c. Prints: country name, capital, population, region, and the list of official languages  
d. Handles the case where the country name is not found (status 404)  

*Hint: languages are stored under `country["languages"]` as a dictionary mapping  
language codes to names, e.g., `{"eng": "English", "fra": "French"}`.*

In [ ]:
# TODO — Exercise 1

<div style="background:#d4edda;border-left:6px solid #28a745;padding:10px 16px;border-radius:4px;margin:10px 0;">
<strong>&#x2705; INSTRUCTOR SOLUTION</strong>
</div>

### Solution for Exercise 1

In [ ]:
# ══════════════════════════════════════════════════════════════
# INSTRUCTOR SOLUTION — Exercise 1
# ══════════════════════════════════════════════════════════════
# Task: prompt for a country name, retrieve data from restcountries.com,
# print name / capital / population / region / languages, handle 404.

country_name = input("Enter a country name: ")
url = f"https://restcountries.com/v3.1/name/{country_name}"
response = requests.get(url)

if response.status_code == 404:
    print(f"Country '{country_name}' not found.")
elif response.status_code == 200:
    country = response.json()[0]
    # languages is a dict mapping code -> name, e.g. {"eng": "English"}
    languages = ", ".join(country["languages"].values())
    print(f"Country:    {country['name']['common']}")
    print(f"Capital:    {country['capital'][0]}")
    print(f"Population: {country['population']:,}")
    print(f"Region:     {country['region']}")
    print(f"Languages:  {languages}")
else:
    print(f"Request failed: HTTP {response.status_code}")


### Exercise 2 — API Key and Query Parameters

Using the OpenWeatherMap API:

a. Query current weather for New York, NY in **metric** units  
b. Extract and print: city name, temperature (°C), humidity (%), wind speed (m/s),
   and weather description  
c. Modify the request to use **latitude and longitude** instead of a city name  
   *(Replace the `q` parameter with `lat` and `lon`.
   See the [OpenWeatherMap docs](http://openweathermap.org/current#geo).)*  
d. Retrieve the weather for **San Francisco, CA** using the city name parameter

In [ ]:
# TODO — Exercise 2a and 2b

In [ ]:
# TODO — Exercise 2c  (lat/lon)

In [ ]:
# TODO — Exercise 2d  (San Francisco)

<div style="background:#d4edda;border-left:6px solid #28a745;padding:10px 16px;border-radius:4px;margin:10px 0;">
<strong>&#x2705; INSTRUCTOR SOLUTION</strong>
</div>

### Solution for Exercise 2

In [ ]:
# ══════════════════════════════════════════════════════════════
# INSTRUCTOR SOLUTION — Exercise 2
# ══════════════════════════════════════════════════════════════
# All four parts are consolidated here.

api_key = userdata.get("openweathermapAPIkey")
owm_url = "http://api.openweathermap.org/data/2.5/weather"

# ── Parts a and b: New York by city name, metric units ────────
parameters = {
    "q":     "New York, NY, USA",
    "units": "metric",
    "appid": api_key
}
response = requests.get(owm_url, params=parameters)
data = response.json()

print("── New York (city name, metric) ──")
print(f"City:        {data['name']}")
print(f"Temperature: {data['main']['temp']} \u00b0C")
print(f"Humidity:    {data['main']['humidity']} %")
print(f"Wind speed:  {data['wind']['speed']} m/s")
print(f"Description: {data['weather'][0]['description']}")

# ── Part c: New York by lat/lon ───────────────────────────────
# Replace the 'q' key with 'lat' and 'lon'; all other parameters unchanged.
parameters_latlon = {
    "lat":   40.7128,
    "lon":  -74.0060,
    "units": "metric",
    "appid": api_key
}
response = requests.get(owm_url, params=parameters_latlon)
data = response.json()

print("\n── New York (lat/lon, metric) ──")
print(f"City:        {data['name']}")
print(f"Temperature: {data['main']['temp']} \u00b0C")
print(f"Humidity:    {data['main']['humidity']} %")
print(f"Wind speed:  {data['wind']['speed']} m/s")
print(f"Description: {data['weather'][0]['description']}")

# ── Part d: San Francisco by city name ───────────────────────
parameters_sf = {
    "q":     "San Francisco, CA, USA",
    "units": "metric",
    "appid": api_key
}
response = requests.get(owm_url, params=parameters_sf)
data = response.json()

print("\n── San Francisco (city name, metric) ──")
print(f"City:        {data['name']}")
print(f"Temperature: {data['main']['temp']} \u00b0C")
print(f"Humidity:    {data['main']['humidity']} %")
print(f"Wind speed:  {data['wind']['speed']} m/s")
print(f"Description: {data['weather'][0]['description']}")


### Exercise 3 — Chaining Two APIs

Write a single block of code that:

1. Uses the ipstack API to look up the geographic location of IP address `8.8.8.8`
2. Extracts the latitude and longitude from the ipstack response
3. Uses those coordinates (with `lat` / `lon` parameters) to query OpenWeatherMap
   for the current weather at that location
4. Prints a summary: IP address, city, country, temperature (°F), and weather description

This exercise demonstrates a common real-world pattern: the output of one API call
becomes the input of the next.

In [ ]:
# TODO — Exercise 3

<div style="background:#d4edda;border-left:6px solid #28a745;padding:10px 16px;border-radius:4px;margin:10px 0;">
<strong>&#x2705; INSTRUCTOR SOLUTION</strong>
</div>

### Solution for Exercise 3

In [ ]:
# ══════════════════════════════════════════════════════════════
# INSTRUCTOR SOLUTION — Exercise 3
# ══════════════════════════════════════════════════════════════
# Demonstrates API chaining: ipstack output feeds OpenWeatherMap input.

ipstack_key = userdata.get("ipstackAPIkey")
owm_key     = userdata.get("openweathermapAPIkey")
target_ip   = "8.8.8.8"

# Step 1: Geolocate the IP address using the function defined earlier.
location = get_ip_location(target_ip, ipstack_key)
lat     = location["latitude"]
lon     = location["longitude"]
city    = location.get("city", "Unknown")
country = location.get("country_name", "Unknown")

# Step 2: Use coordinates returned by ipstack to query OpenWeatherMap.
owm_url    = "http://api.openweathermap.org/data/2.5/weather"
parameters = {
    "lat":   lat,          # ← output of ipstack becomes input here
    "lon":   lon,
    "units": "imperial",
    "appid": owm_key
}
response = requests.get(owm_url, params=parameters)
weather  = response.json()

# Step 3: Print combined summary.
print(f"IP address:  {target_ip}")
print(f"City:        {city}, {country}")
print(f"Temperature: {weather['main']['temp']} \u00b0F")
print(f"Description: {weather['weather'][0]['description']}")


---

> *Submit your solutions on Brightspace. Download your completed notebook as an `.ipynb` file
> (File → Download → Download .ipynb) and upload it to the Module 7 assignment.
> The Brightspace assignment page has full submission details.*

---

## Key Terms

| Term | Definition |
|---|---|
| **API** | A documented contract defining how one program requests services from another |
| **Web API** | An API accessible over the internet using HTTP |
| **REST** | Representational State Transfer — an architectural style for web APIs using HTTP methods and resource-based URLs |
| **Endpoint** | A specific URL at which an API service is accessible |
| **HTTP GET** | The HTTP method used to retrieve data without modifying server state |
| **Status code** | An integer in the HTTP response: 200 = success; 4xx = client error; 5xx = server error |
| **JSON** | JavaScript Object Notation — a text format for structured data that maps to Python dicts and lists |
| **API key** | A credential string passed with requests to authenticate the caller and enforce rate limits |
| **Rate limiting** | A restriction on the number of API requests allowed per unit of time |
| **Query parameter** | A key-value pair appended to a URL (after `?`) that passes arguments to an endpoint |
| **`requests` library** | A Python library for sending HTTP requests and parsing responses |
| **`response.json()`** | Parses the JSON response body and returns a Python dictionary or list |
| **Geolocation** | Determining a physical location (lat/lon) from digital data such as an IP address |
| **API chaining** | Using the output of one API call as the input to a second API call |
| **SOA** | Service-Oriented Architecture — a design pattern in which applications are composed of loosely coupled services communicating over a network |



---

